In [1]:
import pandas as pd
import numpy as np

In [2]:
data = 'housing_data_CDMX_v2.csv'

df = pd.read_csv('housing_data_CDMX_v2.csv')

# Initial Data Inspection

### Dimension of the dataset

In [3]:
df.shape

(18234, 13)

The dataset consists of 18,234 instances (rows) and 13 variables (columns)

### Preview the dataset


In [4]:
df.head(5)

,property_type,places,lat-lon,price,currency,price_aprox_local_currency,price_aprox_usd,surface_total_in_m2,surface_covered_in_m2,price_usd_per_m2,price_per_m2,lat,lon
0,apartment,MiguelHidalgo,"23.634501,-102.552788",5500000.0,MXN,5450245.50,289775.66,54.0,54.0,5366.215926,101851.85190,23.634501,-102.552788
1,house,Iztapalapa,"19.31033,-99.068557",1512000.0,MXN,1498321.97,79661.96,80.0,80.0,995.774500,18900.00000,19.310330,-99.068557
2,apartment,Tlalpan,"19.279771,-99.234597",926667.0,MXN,918284.00,48822.82,100.0,100.0,488.228200,9266.67000,19.279771,-99.234597
3,apartment,MiguelHidalgo,"23.634501,-102.552788",6410000.0,MXN,6352013.39,337720.36,135.0,135.0,2501.632296,47481.48148,23.634501,-102.552788
4,apartment,MiguelHidalgo,"19.432657,-99.177444",4416000.0,MXN,4376051.62,232663.51,87.0,87.0,2674.293218,50758.62069,19.432657,-99.177444


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 18234 entries, 0 to 18233
Data columns (total 13 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   property_type               18234 non-null  str    
 1   places                      18234 non-null  str    
 2   lat-lon                     18234 non-null  str    
 3   price                       18234 non-null  float64
 4   currency                    18234 non-null  str    
 5   price_aprox_local_currency  18234 non-null  float64
 6   price_aprox_usd             18234 non-null  float64
 7   surface_total_in_m2         18234 non-null  float64
 8   surface_covered_in_m2       18234 non-null  float64
 9   price_usd_per_m2            18234 non-null  float64
 10  price_per_m2                18234 non-null  float64
 11  lat                         18234 non-null  float64
 12  lon                         18234 non-null  float64
dtypes: float64(9), str(4)
memory usage: 2.6 MB

### with df.info(), we get a complete structural summary of the dataframe.

**Categorical and Location Variables (Text)**:

- property_type: Type of listed property (apartment, house, PH (Horizontal Property , commercial space).
- places: Administrative location or geographic hierarchy of the property (province/state, city, or neighborhood).
- lat-lon: Geographic coordinates combined into a single text string.
- currency: Original currency in which the property price was listed (MXN, USD).

**Numerical Price Variables**:

- price: Original listed sale price for the property.
- price_aprox_local_currency: Approximate price converted to the corresponding local currency.
- price_aprox_usd: Approximate price standardized in U.S. dollars (USD).
- price_usd_per_m2: Price per square meter calculated in U.S. dollars.
- price_per_m2: Price per square meter in the original listing currency.

**Numerical Area and Coordinate Variables**:

- surface_total_in_m2: Total area of the property measured in square meters (m²).
- surface_covered_in_m2: Covered or built-up area measured in square meters (m²).
- lat: Geographic latitude in decimal format.
- lon: Geographic longitude in decimal format.


### Missing Values

In [6]:
df.isnull().sum()

property_type                 0
places                        0
lat-lon                       0
price                         0
currency                      0
price_aprox_local_currency    0
price_aprox_usd               0
surface_total_in_m2           0
surface_covered_in_m2         0
price_usd_per_m2              0
price_per_m2                  0
lat                           0
lon                           0
dtype: int64

The above command shows that there are no missing values in the dataset.

# Initial Exploratory Data Analysis

- We will verify if there are properties listed more than once using df.duplicated().sum().
  
- The lat-lon column contains the exact same information as lat and lon separately, so we will drop it to simplify the analysis.

- We will check with df.describe() that there are no areas with 0 square meters, unrealistic prices, or cases where surface_covered_in_m2 is greater than surface_total_in_m2.

### Property Type Distribution before Cleaning

# Data Cleaning

### Identifying and Removing Exact Duplicate Rows

In [7]:
df.duplicated().sum()

np.int64(2953)

This code checks each row in the DataFrame to identify duplicate rows. 
The result shows a single integer representing 2,953 of duplicate rows in the DataFrame

In [8]:
df = df.drop_duplicates().reset_index(drop=True)

After removing duplicates, this code resets the DataFrame's index to a default integer index 

During the first data analysis, although no null values were found, **2,953 duplicate records were detected**. 

These were removed to ensure the uniqueness of each property, resulting in a clean sample of **15,281 properties**.

### Identifying Partial Duplicate Rows by Location and Surface

In [9]:
key_columns = ['property_type', 'lat', 'lon', 'surface_total_in_m2', 'surface_covered_in_m2']
partial_duplicates = df.duplicated(subset=key_columns).sum()
print(f"Partial duplicates found: {partial_duplicates}")

Partial duplicates found: 2032


### Analyzing Partial Duplicate Rows by Property Type


In [10]:
partial_dup_df = df[df.duplicated(subset=key_columns, keep=False)]
print("Partial duplicates found by property type:")
partial_dup_df['property_type'].value_counts()

Partial duplicates found by property type:


property_type
apartment    2927
house         488
store          15
Name: count, dtype: int64

### Selective Deduplication by Property Type, Location, Surface, and Price

In [11]:
# Separate apartments from other property types
apartments = df[df['property_type'] == 'apartment']
others = df[df['property_type'] != 'apartment'] 

This separation allows for different deduplication strategies to be applied to apartments versus other property types in the subsequent code.

`apartments =` - The filtered DataFrame containing only apartment properties is assigned to a new variable called `apartments`.

`others =` - The filtered DataFrame containing all non-apartment properties is assigned to a new variable called `others`.

In [12]:
# Drop duplicated Apartments only if location, surface AND price are identical
# This includes property type, location (lat/lon), surface areas, and price.
apt_columns = ['property_type', 'lat', 'lon', 'surface_total_in_m2', 'surface_covered_in_m2', 'price_aprox_usd']
apartments_clean = apartments.drop_duplicates(subset=apt_columns, keep='first')

`subset=apt_columns` - Only considers the columns listed in 'apt_columns' when checking for duplicates, rather than comparing all columns.

`keep='first'` - When duplicates are found, this parameter keeps the first occurrence and removes subsequent duplicates.

`apartments_clean =` Assigns the resulting DataFrame (with duplicates removed) to a new variable called `apartments_clean`.

In [13]:
# Drop duplicated Houses, Storers or Horizontal Properties if location and surface are identical
other_columns = ['property_type', 'lat', 'lon', 'surface_total_in_m2', 'surface_covered_in_m2']
others_clean = others.drop_duplicates(subset=other_columns, keep='first')

`other_columns` creates a list containing five column names that will be used to identify duplicate entries for non-apartment properties.

`others.drop_duplicates(subset=other_columns, keep='first')` removes duplicate rows from the `others` DataFrame where all five specified columns have identical values.

`others_clean =` Assigns the resulting DataFrame (with duplicates removed) to a new variable called `others_clean`.


In [14]:
# Combine into the clean dataframe
df = pd.concat([apartments_clean, others_clean], ignore_index=True) # This function combine `apartments_clean` and `others_clean`

#  Calculate and display how many apartment records were removed by subtracting the cleaned count from the original count.
print(f"Apartments removed: {len(apartments) - len(apartments_clean)}")

# To show how many non-apartment property records were removed during the cleaning process.
print(f"Other properties removed: {len(others) - len(others_clean)}")

# To display the final dimensions (rows, columns) of the combined cleaned DataFrame.
print(f"Dataset shape after selective deduplication: {df.shape}")


Apartments removed: 23
Other properties removed: 272
Dataset shape after selective deduplication: (14986, 13)


A Multi-Step Deduplication Strategy was used because a single filter for distinct real estate properties, such as different apartments within the same building can legitimately share identical coordinates and floor plans. Instead, a progressive workflow was followed: 

- First, **exact duplicate rows** were removed to eliminate identical reposts without risk.
  
- Second, **partial duplicates** were identified by location and surface area and broken down **by property type** to distinguish vertical buildings from standalone properties.

  
- Finally, a two-tier **selective deduplication** was applied: for **apartments**, records were dropped only when sharing identical coordinates, surface areas, and standardized USD price (23 duplicate postings removed), whereas for **houses, stores, and PH (Propiedades Horizontales) units**, matching coordinates and dimensions alone were used to filter out multi-agency reposts (272 duplicates removed), consolidating the dataset to **14,986 unique properties**.


### Identifying Properties outside CDMX boundaries

### Geographic Filtering and Spatial Validation

Prior to performing any exploratory analysis or training machine learning models, it is essential to validate the spatial integrity of the dataset. Real estate listings often contain severe data-entry anomalies; a common issue occurs when systems cannot locate a property's specific street address and automatically assign a default coordinate representing the country's exact geographical center (approximately 23.63, -102.55, located in Zacatecas). Left untreated, these massive positional errors would heavily distort any neighborhood-based metrics, spatial plots, or distance-based model predictions. 

To ensure geographic validity, spatial boundaries for Mexico City were established based on official coordinates provided by the National Institute of Statistics and Geography (INEGI, 2021), removing out-of-bounds anomalies outside the capital's perimeter. Officially, Mexico City spans from 19.04° to 19.59° North latitude, and from -99.36° to -98.94° West longitude. By applying a slight tolerance margin to account for rounding and liminal suburban properties, we establish strict boundary constants ("lat" between 19.0 and 19.6; "lon" between -99.4 and -98.9). The following code flags any records falling outside this window as erroneous, purges them from the dataset, and ensures all subsequent analysis is built exclusively on geographically valid Mexico City data.



In [15]:
# To define approximate geographic boundaries for Mexico City (CDMX)
lat_min, lat_max = 19.0, 19.6
lon_min, lon_max = -99.4, -98.9

# To Identify properties outside CDMX boundaries
outside_cdmx = df[(df['lat'] < lat_min) | (df['lat'] > lat_max) | (df['lon'] < lon_min) | (df['lon'] > lon_max)]

print(f"Properties outside CDMX found: {len(outside_cdmx)}")
display(outside_cdmx[['property_type', 'places', 'lat', 'lon', 'price_aprox_usd']].head())

# Filter the dataframe to keep only valid CDMX coordinates
df = df[(df['lat'].between(lat_min, lat_max)) & (df['lon'].between(lon_min, lon_max))].reset_index(drop=True)

print(f"Dataset shape after geographic filter: {df.shape}")

Properties outside CDMX found: 185


,property_type,places,lat,lon,price_aprox_usd
0,apartment,MiguelHidalgo,23.634501,-102.552788,289775.66
2,apartment,MiguelHidalgo,23.634501,-102.552788,337720.36
5,apartment,BenitoJuarez,23.634501,-102.552788,94022.66
64,apartment,BenitoJuarez,23.634501,-102.552788,158059.45
113,apartment,Cuauhtemoc,23.634501,-102.552788,62170.05


Dataset shape after geographic filter: (14801, 13)


This filter successfully detected and removed **185 records** featuring default, out-of-bounds country-center coordinates (e.g., 23.63, -102.55), ensuring the geographic integrity of the dataset.

### Quantitative Integrity Revision

Before cleaning the numeric data, we will run a quick diagnostic check to identify data-entry errors and format inconsistencies:

- **Summary Statistics (df.describe().T)**: We check the minimum and maximum values to detect extreme, unrealistic outliers in prices and sizes.
- **Currency Check**: We count the listings in Pesos (MXN) versus Dollars (USD) to confirm if the currencies are mixed and need standardization.
- **Logical Consistency Check**: We count how many records have a physical impossibility, where the covered built area (surface_covered_in_m2) is larger than the total land area (surface_total_in_m2).

The results of this audit will define the specific thresholds used in the next cleaning step.


In [16]:
# Summary statistics and check for logical inconsistencies
display(df.describe().T)

print("Currencies:", df['currency'].value_counts().to_dict())
print("Covered > Total surface:", (df['surface_covered_in_m2'] > df['surface_total_in_m2']).sum())

,count,mean,std,min,25%,50%,75%,max
price,14801.0,3.746419e+06,5.926947e+06,7000.000000,1.009000e+06,1.990666e+06,4.094000e+06,2.337267e+08
price_aprox_local_currency,14801.0,5.482942e+06,1.033780e+07,109030.610000,1.080234e+06,2.279194e+06,4.954768e+06,3.364841e+08
price_aprox_usd,14801.0,2.915140e+05,5.496347e+05,5796.880000,5.743327e+04,1.211789e+05,2.634324e+05,1.789000e+07
surface_total_in_m2,14801.0,4.975025e+03,5.408346e+05,1.000000,6.900000e+01,9.800000e+01,2.000000e+02,6.574800e+07
surface_covered_in_m2,14801.0,4.947112e+03,5.408333e+05,1.000000,6.900000e+01,9.800000e+01,2.120000e+02,6.574800e+07
price_usd_per_m2,14801.0,1.505663e+03,2.293944e+03,0.000600,6.587372e+02,1.071099e+03,1.898517e+03,1.759177e+05
price_per_m2,14801.0,2.430818e+04,4.394172e+04,0.011377,1.112471e+04,1.782836e+04,3.200000e+04,3.338953e+06
lat,14801.0,1.938568e+01,5.750544e-02,19.185852,1.935021e+01,1.938725e+01,1.942301e+01,1.957032e+01
lon,14801.0,-9.916472e+01,5.338701e-02,-99.333807,-9.919475e+01,-9.916319e+01,-9.913435e+01,-9.897008e+01


Currencies: {'MXN': 13892, 'USD': 909}
Covered > Total surface: 869


In [17]:
# Remove records where covered surface exceeds total surface
df = df[df['surface_covered_in_m2'] <= df['surface_total_in_m2']].reset_index(drop=True)

# Drop redundant or mixed-currency columns
cols_to_drop = ['lat-lon', 'price', 'currency', 'price_per_m2']
df = df.drop(columns=cols_to_drop)

print(f"Dataset shape after surface and column cleanup: {df.shape}")
display(df.describe().T[['min', '50%', 'max']])


Dataset shape after surface and column cleanup: (13932, 9)


,min,50%,max
price_aprox_local_currency,109030.610000,2.149229e+06,3.364841e+08
price_aprox_usd,5796.880000,1.142690e+05,1.789000e+07
surface_total_in_m2,1.000000,9.200000e+01,6.574800e+07
surface_covered_in_m2,1.000000,9.100000e+01,6.574800e+07
price_usd_per_m2,0.000600,1.036510e+03,1.759177e+05
lat,19.185852,1.938851e+01,1.957032e+01
lon,-99.333807,-9.916305e+01,-9.897008e+01


### Outlier Detection in Summary Statistics

Although the median (50%) values align with normal Mexico City real estate metrics ($114,269 USD, 92 m², and $1,037 USD/m²), the extreme boundaries reveal severe data-entry anomalies. Specifically, surface areas range from an impossible minimum of 1 m² to an unrealistic maximum of 65.7 million m², which in turn distorts unit prices (price_usd_per_m2) from $0.0006 up to $175,918 USD/m², alongside abnormally low listing prices starting at $5,797 USD. These extreme values justify applying domain-specific market thresholds in the next step to filter out capture errors.

In [18]:
# Apply realistic real estate thresholds for Mexico City (CDMX)
df = df[(df['surface_total_in_m2'].between(20, 2500)) & (df['surface_covered_in_m2'].between(20, 2500)) &
    (df['price_usd_per_m2'].between(150, 10000)) & (df['price_aprox_usd'] >= 15000)].reset_index(drop=True)

print(f"Final dataset shape after outlier removal: {df.shape}")
display(df.describe().T[['min', '50%', 'max']])


Final dataset shape after outlier removal: (13605, 9)


,min,50%,max
price_aprox_local_currency,284570.150000,2.188026e+06,1.238986e+08
price_aprox_usd,15129.870000,1.163318e+05,6.587372e+06
surface_total_in_m2,20.000000,9.200000e+01,2.450000e+03
surface_covered_in_m2,20.000000,9.100000e+01,2.236000e+03
price_usd_per_m2,150.156475,1.053730e+03,9.861789e+03
lat,19.185852,1.938828e+01,1.957032e+01
lon,-99.333807,-9.916361e+01,-9.897008e+01


In [19]:
print(df['property_type'].value_counts())

property_type
apartment    10055
house         3430
store          106
PH              14
Name: count, dtype: int64


### Data Cleaning Summary

Starting from **18,234 raw records** across 13 variables, a systematic five-step cleaning pipeline was executed:

- **Exact Duplicates**: Removed **2,953** identical rows (`15,281` remaining).
- **Selective Partial Duplicates**: Applied a two-tier deduplication strategy—removing **23** duplicate apartments sharing identical coordinates, surface areas, and standardized USD prices, along with **272** duplicate houses, stores, and PH units sharing matching locations and dimensions (`14,986` remaining).
- **Spatial Validation**: Filtered out **185** records with coordinates outside Mexico City's official geographic bounding box (14,801 remaining).
- **Metric Consistency**: Removed **869** records where covered surface exceeded total surface and dropped redundant or mixed-currency columns (13,932 remaining).
- **Outlier Filtering**: Removed **327** extreme capture anomalies (e.g., 1 m² or 65.7M m² listings) using realistic CDMX market thresholds.
  
- **Final Clean Dataset** of **13,605 properties** and **9 standardized features**.



In [20]:
df.to_csv('housing_data_CDMX_clean.csv', index=False)